# Pump Scenario Sample Generator & Visualiser

Runs all **10 pump scenarios** (1 normal + 3 failures + 6 decoys) serially using NASA's **progpy CentrifugalPump** model with:
- Cycling voltage load (1h period) for natural signal variation
- Per-signal `process_noise` and `measurement_noise` for realistic sensor jitter
- Wear rates set via `x0` initial state (`wA=0.01` → failure ~6h)

| Group | Failure mode | Discriminating signals | Hard negatives |
|-------|-------------|----------------------|----------------|
| Bearing | `wA=0.01` | shaft speed ↓ **AND** temperature ↑ | setpoint step/ramp (speed ↓, temp flat) |
| Impeller | `wA=0.01` | flow_out ↓ **AND** Qin/Qout ratio > 1 | back-pressure step/ramp (ratio ≈ 1) |
| Seal | `wA=0.01` | flow_out ↓ **AND** flow_in stable (gap grows) | flow-reduction step/ramp (gap ≈ 0) |

In [ ]:
import sys, importlib, warnings
warnings.filterwarnings('ignore')

# Make sample_generator importable and reload if already loaded
if "sample_generator" in sys.modules:
    importlib.reload(sys.modules["sample_generator"])
from sample_generator import build_scenarios, run_all

## Run all 9 simulations
Each scenario runs ~360 steps (6 h at 1 sample/min). Faulty scenarios stop early at the failure threshold.

In [ ]:
scenarios = build_scenarios()
results = run_all(scenarios)

In [ ]:
# Quick summary table
import pandas as pd

rows = []
for name, df in results.items():
    vc = df["label"].value_counts()
    rows.append({
        "scenario": name,
        "rows": len(df),
        "NORMAL": vc.get("NORMAL", 0),
        "PRE_FAILURE": vc.get("PRE_FAILURE", 0),
        "FAILURE": vc.get("FAILURE", 0),
    })
pd.DataFrame(rows).set_index("scenario")

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.dpi"] = 150

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sample_generator import (
    COLOURS, LABEL_ALPHA, LABEL_COLOUR,
    _shade_labels, _plot_line, _finish_ax,
)

## Plot 0 — Healthy baseline
All signals stable, every row labelled NORMAL. This is what "nothing is wrong" looks like.

In [ ]:
df = results["normal"]

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
fig.suptitle("Healthy pump — baseline signals (all NORMAL)",
             fontsize=12, fontweight="bold")

pairs = [
    ("shaft_speed_rads", "Shaft speed (rad/s)"),
    ("fluid_temp_K",     "Temperature (K)"),
    ("flow_out_m3s",     "Flow out (m\u00b3/s)"),
    ("flow_in_m3s",      "Flow in (m\u00b3/s)"),
    ("pump_speed_rads",  "Pump speed (rad/s)"),
]

for idx, (col, label) in enumerate(pairs):
    ax = axes[idx // 3][idx % 3]
    if col in df.columns:
        ax.plot(df["time_h"], df[col], color=COLOURS["normal"], linewidth=1.5)
    ax.set_title(label, fontsize=9, fontweight="bold")
    ax.set_xlabel("Time (hours)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.25, linewidth=0.5)

# Last axis: label distribution bar
ax = axes[1][2]
counts = df["label"].value_counts()
ax.bar(counts.index, counts.values, color=["#1D9E75"])
ax.set_title("Label distribution", fontsize=9, fontweight="bold")
ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()

## Plot 1 — Thrust bearing wear vs high-load decoys

**Both** show Tt rising — but the failure rises **faster** than the operating conditions explain.

- **Failure:** Tt climbs with no speed increase — excess heat from friction (`rThrust * w^2`)
- **Hard negative:** Higher voltage → speed ↑ → Tt rises too, but proportionally — no excess heat

The discrimination problem: you can't just threshold on "Tt is rising" — you need to check whether the rate of rise is explained by the current load.

In [ ]:
from sample_generator import plot_bearing_group, plot_impeller_group, plot_seal_group

bearing_scenarios = [
    ("pump_bearing_wear",   "Thrust bearing wear (FAILURE)"),
    ("decoy_highload_step", "High-load step (NORMAL decoy)"),
    ("decoy_highload_ramp", "High-load ramp (NORMAL decoy)"),
]
normal_df = results["normal"]

fig, axes = plt.subplots(3, 2, figsize=(12, 9))
fig.suptitle(
    "Thrust bearing wear vs high-load decoys\n"
    "Both show Tt rising \u2014 but failure rises FASTER than load explains\n"
    "Decoy: Tt rises proportionally to speed increase (no excess heat)",
    fontsize=11, fontweight="bold", y=0.99
)

for row, (name, title) in enumerate(bearing_scenarios):
    df     = results[name]
    colour = COLOURS["failure"] if "wear" in name else COLOURS["decoy"]

    # Left: thrust bearing temperature
    ax = axes[row, 0]
    _shade_labels(ax, df)
    _plot_line(ax, df, "thrust_bearing_K", colour, "Tt (this scenario)")
    ax.plot(normal_df["time_h"], normal_df["thrust_bearing_K"],
            color="#999", linewidth=1.0, linestyle="--", label="Tt (healthy)")
    _finish_ax(ax, "Thrust bearing Tt (K)", title)

    # Right: shaft speed
    ax = axes[row, 1]
    _shade_labels(ax, df)
    _plot_line(ax, df, "shaft_speed_rads", colour, "shaft speed")
    ax.plot(normal_df["time_h"], normal_df["shaft_speed_rads"],
            color="#999", linewidth=1.0, linestyle="--", label="w (healthy)")
    _finish_ax(ax, "Shaft speed (rad/s)", title)

axes[0, 0].set_title("thrust_bearing_K (Tt)\n" + axes[0, 0].get_title(),
                      fontsize=9, fontweight="bold", pad=4)
axes[0, 1].set_title("shaft_speed_rads\n" + axes[0, 1].get_title(),
                      fontsize=9, fontweight="bold", pad=4)

fig.text(0.5, 0.01,
         "Failure: Tt rises with NO speed increase (excess friction)  |  "
         "Decoy: Tt rises WITH speed increase (expected load heat)",
         ha="center", fontsize=8, color="#555")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## Plot 2 — Impeller wear vs back-pressure decoys

**Failure signature:** impeller area `A` shrinks from 12.7 to 9.5 as the impeller erodes (`Adot = -wA * Q^2`)

The hard negatives (back-pressure step/ramp) also reduce flow, but `A` stays constant at 12.7 — the degradation is purely operational, not structural.

In [ ]:
impeller_scenarios = [
    ("pump_impeller_wear",       "Impeller wear (FAILURE)"),
    ("decoy_back_pressure_step", "Back pressure step (NORMAL decoy)"),
    ("decoy_back_pressure_ramp", "Back pressure ramp (NORMAL decoy)"),
]

fig, axes = plt.subplots(3, 2, figsize=(12, 9))
fig.suptitle(
    "Impeller wear vs back-pressure decoys\n"
    "Correct rule: A (impeller area) shrinks \u2192 impeller wear\n"
    "Decoys reduce flow via pressure but A stays constant",
    fontsize=11, fontweight="bold", y=0.99
)

for row, (name, title) in enumerate(impeller_scenarios):
    df     = results[name]
    colour = COLOURS["failure"] if "wear" in name else COLOURS["decoy"]

    # Left: impeller area A
    ax = axes[row, 0]
    _shade_labels(ax, df)
    _plot_line(ax, df, "impeller_area_A", colour, "A (impeller area)")
    _finish_ax(ax, "Impeller area A", title, show_legend=False)

    # Right: flow_out
    ax = axes[row, 1]
    _shade_labels(ax, df)
    _plot_line(ax, df, "flow_out_m3s", colour, "flow out")
    _finish_ax(ax, "Flow out (m\u00b3/s)", title, show_legend=False)

axes[0, 0].set_title("impeller_area_A (state)\n" + axes[0,0].get_title(),
                      fontsize=9, fontweight="bold", pad=4)
axes[0, 1].set_title("flow_out_m3s\n" + axes[0,1].get_title(),
                      fontsize=9, fontweight="bold", pad=4)

fig.text(0.5, 0.01,
         "Failure: A shrinks from 12.7 \u2192 9.5 as impeller erodes  |  "
         "Decoys: A stays flat at 12.7 \u2014 flow drop is from pressure, not wear",
         ha="center", fontsize=8, color="#555")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## Plot 3 — Radial bearing wear vs high-load decoys

Same logic as thrust bearing: **both** show Tr rising, but the failure has excess heat not explained by load.

- **Failure:** Tr climbs with no speed increase — friction from degraded radial bearing
- **Hard negative:** Higher load → speed ↑ → Tr rises proportionally

In [ ]:
radial_scenarios = [
    ("pump_radial_wear",             "Radial bearing wear (FAILURE)"),
    ("decoy_radial_highload_step",   "High-load step (NORMAL decoy)"),
    ("decoy_radial_highload_ramp",   "High-load ramp (NORMAL decoy)"),
]

fig, axes = plt.subplots(3, 2, figsize=(12, 9))
fig.suptitle(
    "Radial bearing wear vs high-load decoys\n"
    "Both show Tr rising \u2014 but failure rises FASTER than load explains\n"
    "Decoy: Tr rises proportionally to speed increase (no excess heat)",
    fontsize=11, fontweight="bold", y=0.99
)

for row, (name, title) in enumerate(radial_scenarios):
    df     = results[name]
    colour = COLOURS["failure"] if "wear" in name else COLOURS["decoy"]

    # Left: radial bearing temperature
    ax = axes[row, 0]
    _shade_labels(ax, df)
    _plot_line(ax, df, "radial_bearing_K", colour, "Tr (this scenario)")
    ax.plot(normal_df["time_h"], normal_df["radial_bearing_K"],
            color="#999", linewidth=1.0, linestyle="--", label="Tr (healthy)")
    _finish_ax(ax, "Radial bearing Tr (K)", title)

    # Right: shaft speed
    ax = axes[row, 1]
    _shade_labels(ax, df)
    _plot_line(ax, df, "shaft_speed_rads", colour, "shaft speed")
    ax.plot(normal_df["time_h"], normal_df["shaft_speed_rads"],
            color="#999", linewidth=1.0, linestyle="--", label="w (healthy)")
    _finish_ax(ax, "Shaft speed (rad/s)", title)

axes[0, 0].set_title("radial_bearing_K (Tr)\n" + axes[0,0].get_title(),
                      fontsize=9, fontweight="bold", pad=4)
axes[0, 1].set_title("shaft_speed_rads\n" + axes[0,1].get_title(),
                      fontsize=9, fontweight="bold", pad=4)

fig.text(0.5, 0.01,
         "Failure: Tr rises with NO speed increase (excess friction)  |  "
         "Decoy: Tr rises WITH speed increase (expected load heat)",
         ha="center", fontsize=8, color="#555")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()